# 02: Data Preprocessing & Leakage Prevention

### Objectives:
1. Validate and clip physically impossible sensor readings (pyranometer noise, negative power).
2. Enforce strict chronological partitioning ($70\%$ Train, $15\%$ Validation, $15\%$ Untouched Test).
3. Ensure all transformers and scalers are fitted strictly on the training partition.


In [ ]:
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, '../src')
from data_loader import load_raw_datasets, aggregate_plant_level
from preprocessing import SolarDataCleaner
from model_training import prepare_full_dataset

print('Modules loaded.')


### 1. Physical Sensor Boundary Validation
Physical boundaries for photovoltaic sensors:
- Global Solar Irradiance: $[0.0, 1.6]\ \text{kW/m}^2$
- Ambient Temperature: $[-10.0, 60.0]^\circ\text{C}$
- PV Module Surface Temperature: $[-10.0, 90.0]^\circ\text{C}$
- AC Power Generation: $\ge 0.0\ \text{kW}$


In [ ]:
cleaner = SolarDataCleaner()
train_df, val_df, test_df = prepare_full_dataset()
print(f'Train partition: {len(train_df):,} rows ({train_df["DATE_TIME"].min()} to {train_df["DATE_TIME"].max()})')
print(f'Val partition:   {len(val_df):,} rows ({val_df["DATE_TIME"].min()} to {val_df["DATE_TIME"].max()})')
print(f'Test partition:  {len(test_df):,} rows ({test_df["DATE_TIME"].min()} to {test_df["DATE_TIME"].max()})')


### 2. Verifying Split Integrity and Zero Temporal Overlap
We verify that the maximum timestamp of the training set is strictly earlier than the minimum timestamp of the validation set.


In [ ]:
assert train_df['DATE_TIME'].max() < val_df['DATE_TIME'].min()
assert val_df['DATE_TIME'].max() < test_df['DATE_TIME'].min()
print('Chronological split integrity verified: No temporal overlap or lookahead leakage!')
